# RoboMME-RI：扰动与 RI-Mem

本 notebook 展示 FS+Mod、TD+Mod 和 SimpleMemVLA 的部分真实 clean 成功 → 扰动失败案例，并提供多 baseline 闭环评测入口。

方法和协议以 `Robo_RI_CVPR27.pdf` 为主，RI-Mem 的公式和参数以 supplement §1、§2.3 为准。
下方视频来自既有 `tr-v1/ev-2` 实验，用于定性观察；旧实验的手送方式、扰动时序和预算与论文协议不同，不能将这些视频直接记入新版 10 条件成功率，也没有将它们标成 RI-Mem 增强结果。

In [ ]:
from pathlib import Path
import sys, json, hashlib
from IPython.display import display, Markdown, Video, Image
cwd = Path.cwd().resolve()
PACKAGE = next(p for p in [cwd, *cwd.parents, cwd/'robustness_test/submitcodes'] if (p/'robomme_ri').is_dir())
sys.path.insert(0, str(PACKAGE))
NB = PACKAGE/'notebooks'
cases = json.loads((NB/'cases.json').read_text())
assert all(c['clean_status']=='success' and c['perturbed_status']!='success' for c in cases)
for case in cases:
    assert hashlib.sha256((NB/case['video']).read_bytes()).hexdigest() == case['sha256']
print(f'{len(cases)} 对已核验案例，{len(set(c["method"] for c in cases))} 个方法')

## 十个条件

clean 与各扰动使用相同的官方任务和 seed。每条 episode 仅有一个条件；light 的 bright/dark 和物件的 insert/remove 均为子类型。事件按环境控制步推进，动作块内部同样触发。

In [ ]:
import pandas as pd
from dataclasses import asdict
from robomme_ri import CONDITIONS
display(pd.DataFrame([{'condition':k, **asdict(v)} for k,v in CONDITIONS.items()]))

## 配对案例

视频左侧为 clean，右侧为扰动。上、中、末帧只帮助定位过程；任务失败类别和具体机制仍应结合完整视频、子指令及轨迹判断。S→F 配对说明该运行发生行为变化，单个案例不能证明记忆导致失败。

### 1. perceptual-framesamp-modul / ButtonUnmaskSwap / episode 2

**Instruction:** first press both buttons on the table, then pick up the container hiding the blue cube

事件：`mid_arrive_early`；作用物体：`button_left`；角色：`answer_only`；施动方式：`hand`。
clean：成功（348 步）；扰动：timeout（1485 步）。

[观看 clean / perturbed 配对视频](assets/01_perceptual-framesamp-modul_ButtonUnmaskSwap_ep2.mp4)

![关键帧](assets/01_perceptual-framesamp-modul_ButtonUnmaskSwap_ep2.jpg)


In [ ]:
case = cases[0]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 2. perceptual-tokendrop-modul / ButtonUnmaskSwap / episode 3

**Instruction:** first press both buttons on the table, then pick up the container hiding the green cube, finally pick up another container hiding the blue cube

事件：`on_arrive_sync`；作用物体：`button_left`；角色：`answer_only`；施动方式：`slide`。
clean：成功（759 步）；扰动：fail（326 步）。

[观看 clean / perturbed 配对视频](assets/02_perceptual-tokendrop-modul_ButtonUnmaskSwap_ep3.mp4)

![关键帧](assets/02_perceptual-tokendrop-modul_ButtonUnmaskSwap_ep3.jpg)


In [ ]:
case = cases[1]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 3. perceptual-framesamp-modul / VideoUnmask / episode 0

**Instruction:** watch the video carefully, then pick up the container hiding the green cube

事件：`mid_takeback_early`；作用物体：`bin_0`；角色：`answer`；施动方式：`slide`。
clean：成功（97 步）；扰动：fail（104 步）。

[观看 clean / perturbed 配对视频](assets/03_perceptual-framesamp-modul_VideoUnmask_ep0.mp4)

![关键帧](assets/03_perceptual-framesamp-modul_VideoUnmask_ep0.jpg)


In [ ]:
case = cases[2]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

### 4. simplememvla-robomme / BinFill / episode 10

**Instruction:** put one red cube and three green cubes into the bin, then press the button to stop

事件：`mid_displace`；作用物体：`board_with_hole`；角色：`answer_only`；施动方式：`hand`。
clean：成功（1026 步）；扰动：fail（862 步）。

[观看 clean / perturbed 配对视频](assets/04_simplememvla-robomme_BinFill_ep10.mp4)

![关键帧](assets/04_simplememvla-robomme_BinFill_ep10.jpg)


In [ ]:
case = cases[3]
display(Image(filename=str(NB/case['poster']), width=840))
display(Video(filename=str(NB/case['video']), embed=True, width=840))

## 统一 RI-Mem

SCQ 解析任务并维护当前子目标；RGB 区域组成历史证据索引。AMR 对原策略已经保留的 keys 计算 relevance、irrelevance、observability、temporal validity，再生成有界 attention bias 和 gate。FS+Mod / TD+Mod 在原有 memory attention 读出时使用 bias 并缩放输出；SimpleMemVLA 仅在第 27、31 层的历史 video keys 上添加 bias 和 log(gate)。

三种方法共享查询、证据打分和 35 参数 head 的实现；各自使用独立校准权重。下例仅展示公式输出，不代表增强 rollout 的成功率。

In [ ]:
import numpy as np
from robomme_ri.rimem.head import initial_parameters, numpy_readout
W, beta = initial_parameters()
facts = np.array([[.9,0,.95,-1], [.9,0,0,1], [0,.95,0,0]], dtype=np.float32)
bias, gate = numpy_readout([1,0,0,0,1,1], facts)
display(pd.DataFrame({'evidence':['outdated spatial state','retained identity','explicit distractor'], 'bias':bias}))
print('Parameters:', W.size+beta.size, 'gate:', float(gate))

## 多 baseline 闭环实验

`configs/experiment.json` 默认比较多个 baseline 与三种 RI-Mem 版本，并跨多个任务/seed；`configs/paper.json` 展开至全部 16 任务 × 50 seed × clean 和 10 条件。运行前填写模型路径、各方法运行环境和服务端地址，并完成 RI-Mem 校准。每个方法使用独立进程，避免不同依赖环境互相影响。

没有可行事件窗口的样本单独记录并报告覆盖率，不冒充已完成扰动；技术错误与任务失败分开记录。

In [ ]:
from robomme_ri.cli import load_configuration, grid
from robomme_ri.baselines import METHODS
cfg = load_configuration(PACKAGE/'configs/experiment.json')
display(pd.DataFrame([{'method':m,'label':METHODS[m].label,'category':METHODS[m].category,'ri_mem':METHODS[m].rimem} for m in cfg['methods']]))
planned = list(grid(cfg))
print('Planned closed-loop episodes:', len(planned))
RUN_EVALUATION = False
if RUN_EVALUATION:
    import subprocess
    subprocess.run([sys.executable,str(PACKAGE/'scripts/run_experiment.py'),str(PACKAGE/'configs/experiment.json'),
                    '--output',str(PACKAGE/'outputs/experiment'),'--gpus','0'],check=True)

In [ ]:
from robomme_ri.evaluate import summarize
paths = sorted((PACKAGE/'outputs/experiment').glob('**/rollouts.jsonl'))
if paths:
    rows = [json.loads(line) for p in paths for line in p.read_text().splitlines() if line.strip()]
    display(pd.DataFrame(summarize(rows)))
else:
    print('本提交目录尚无完整新版实验结果；上方展示的是已核验的既有定性案例。')